# Módulo 5 — Orquestación del Pipeline de Retention Risk

**Sesión 3 (parte 1) — Lunes 4 de Mayo, 2026**

Este notebook **orquesta el pipeline construido en el Módulo 6**. La lógica (UDFs, Stored Procedures, modelo BQML) ya vive en BigQuery; aquí decidimos **cuándo y cómo** ejecutarla.

> **Importante: ejecuta primero el notebook del Módulo 6.** Necesitamos los SPs `sp_build_retention_features`, `sp_apply_decision_rule` y la tabla `retention_actions` ya desplegados.

**Lo que vamos a construir:**

```
[Cloud Scheduler: día 1 de mes 06:00] ──┐
                                         ├──→ [Cloud Workflows: retention-pipeline] ──→ [SPs en BQ]
[Eventarc: payroll_*.csv en GCS]   ─────┘                  │
                                                            ├──→ [Pub/Sub: éxito]
                                                            └──→ [Pub/Sub: fallo → alertas]
```

**Estructura del notebook:**

| § | Tema |
|---|------|
| 1 | Setup + verificación de prerequisitos M6 |
| 2 | Comparativa Workflows vs Composer (tabla decisional) |
| 3 | Despliegue de Cloud Workflows YAML |
| 4 | Configurar topics Pub/Sub para status y alertas |
| 5 | Ejecución manual del Workflow + inspección |
| 6 | Cloud Scheduler — cron mensual |
| 7 | Eventarc — alternativa event-driven |
| 8 | Composer DAG equivalente (lectura, no deploy) |
| 9 | Caso de negocio newsvendor — el "porqué" final |
| 10 | Findings honestos + cierre |


---
## 1. Setup + verificación de prerequisitos

Detección de entorno, clientes, y verificación de que el Módulo 6 ya construyó:
- `feature_store_retention.sp_build_retention_features`
- `predictions_retention.sp_apply_decision_rule`
- `predictions_retention.retention_actions`

In [1]:
# Instalar dependencias (descomentar en primera ejecución)
# !pip install google-cloud-bigquery google-cloud-storage google-cloud-pubsub google-cloud-scheduler google-cloud-workflows google-cloud-eventarc python-dotenv

import os
import json
import time
import subprocess
from datetime import datetime, date, timezone
from google.cloud import bigquery, storage, pubsub_v1
from google.api_core.exceptions import NotFound, AlreadyExists, PermissionDenied
import warnings
warnings.filterwarnings('ignore')

# --- Detección de entorno ---
IN_VERTEX_AI = any([
    os.environ.get("DL_ANACONDA_HOME"),
    os.path.exists("/opt/deeplearning/metadata"),
    os.environ.get("GOOGLE_CLOUD_PROJECT"),
])

if IN_VERTEX_AI:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "people-analytics-formacion")
    bq_client = bigquery.Client(project=PROJECT_ID)
    publisher = pubsub_v1.PublisherClient()
    print(f"Entorno: Vertex AI Workbench (ADC)")
else:
    from dotenv import load_dotenv
    from google.oauth2 import service_account
    load_dotenv()
    PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "people-analytics-formacion")
    creds_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "service-account.json")
    if os.path.exists(creds_path):
        credentials = service_account.Credentials.from_service_account_file(creds_path)
        bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
        publisher = pubsub_v1.PublisherClient(credentials=credentials)
    else:
        bq_client = bigquery.Client(project=PROJECT_ID)
        publisher = pubsub_v1.PublisherClient()
    print(f"Entorno: Local")

REGION = "europe-southwest1"
WORKFLOW_NAME = "retention-pipeline"
SCHEDULER_JOB_NAME = "retention-pipeline-monthly"
EVENTARC_TRIGGER_NAME = "retention-on-payroll-arrival"
TOPIC_STATUS = "retention-pipeline-status"
TOPIC_ALERTS = "retention-pipeline-alerts"

# Service account para orquestación
SA_WORKFLOWS_NAME = "sa-workflows-retention"
SA_WORKFLOWS_EMAIL = f"{SA_WORKFLOWS_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"

print(f"Proyecto:  {PROJECT_ID}")
print(f"Región:    {REGION}")
print(f"Workflow:  {WORKFLOW_NAME}")
print(f"SA:        {SA_WORKFLOWS_EMAIL}")


/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.pubsub_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.pubsub_v1 past that date.
  warnings.warn(message, FutureWarning)


Entorno: Vertex AI Workbench (ADC)
Proyecto:  project-9176af0b-ecb3-4050-859
Región:    europe-southwest1
Workflow:  retention-pipeline
SA:        sa-workflows-retention@project-9176af0b-ecb3-4050-859.iam.gserviceaccount.com


In [2]:
# Verificar prerequisitos de M6 — abortar si no están
PREREQS = [
    ("dataset",   f"{PROJECT_ID}.feature_store_retention"),
    ("dataset",   f"{PROJECT_ID}.predictions_retention"),
    ("dataset",   f"{PROJECT_ID}.pipeline_runs"),
    ("table",     f"{PROJECT_ID}.feature_store_retention.features"),
    ("table",     f"{PROJECT_ID}.predictions_retention.retention_actions"),
    ("table",     f"{PROJECT_ID}.pipeline_runs.retention_pipeline_runs"),
    ("routine",   f"{PROJECT_ID}.feature_store_retention.sp_build_retention_features"),
    ("routine",   f"{PROJECT_ID}.predictions_retention.sp_apply_decision_rule"),
]

print(f"{'Tipo':<10} {'Recurso':<70} {'Estado':<15}")
print("-" * 100)

todas_ok = True
for kind, full_id in PREREQS:
    try:
        if kind == "dataset":
            bq_client.get_dataset(full_id)
        elif kind == "table":
            bq_client.get_table(full_id)
        elif kind == "routine":
            ds_id, routine_name = full_id.rsplit('.', 1)
            bq_client.get_routine(f"{ds_id}.{routine_name}")
        print(f"{kind:<10} {full_id:<70} {'OK':<15}")
    except NotFound:
        print(f"{kind:<10} {full_id:<70} {'FALTA':<15}")
        todas_ok = False

if not todas_ok:
    raise RuntimeError("Faltan recursos del M6. Ejecuta primero el notebook del Módulo 6.")
print()
print("OK — todos los prerequisitos del M6 están desplegados.")


Tipo       Recurso                                                                Estado         
----------------------------------------------------------------------------------------------------
dataset    project-9176af0b-ecb3-4050-859.feature_store_retention                 OK             
dataset    project-9176af0b-ecb3-4050-859.predictions_retention                   OK             
dataset    project-9176af0b-ecb3-4050-859.pipeline_runs                           OK             
table      project-9176af0b-ecb3-4050-859.feature_store_retention.features        OK             
table      project-9176af0b-ecb3-4050-859.predictions_retention.retention_actions OK             
table      project-9176af0b-ecb3-4050-859.pipeline_runs.retention_pipeline_runs   OK             
routine    project-9176af0b-ecb3-4050-859.feature_store_retention.sp_build_retention_features OK             
routine    project-9176af0b-ecb3-4050-859.predictions_retention.sp_apply_decision_rule OK             


In [3]:
# Habilitar APIs necesarias para orquestación
APIS = [
    "workflows.googleapis.com",
    "workflowexecutions.googleapis.com",
    "cloudscheduler.googleapis.com",
    "eventarc.googleapis.com",
    "pubsub.googleapis.com",
    "iam.googleapis.com",
]

for api in APIS:
    print(f"Habilitando {api} ...", end=" ")
    r = subprocess.run(["gcloud", "services", "enable", api, f"--project={PROJECT_ID}"],
                       capture_output=True, text=True)
    print("OK" if r.returncode == 0 else f"FALLO: {r.stderr.strip()[:200]}")

print("\nEsperando 20s para propagación...")
time.sleep(20)


Habilitando workflows.googleapis.com ... OK
Habilitando workflowexecutions.googleapis.com ... OK
Habilitando cloudscheduler.googleapis.com ... OK
Habilitando eventarc.googleapis.com ... OK
Habilitando pubsub.googleapis.com ... OK
Habilitando iam.googleapis.com ... OK

Esperando 20s para propagación...


In [4]:
# Crear service account para Workflows con los roles mínimos necesarios
def crear_sa(name, display_name, project):
    r = subprocess.run([
        "gcloud", "iam", "service-accounts", "create", name,
        f"--display-name={display_name}",
        f"--project={project}",
    ], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  SA creada: {name}")
    elif "already exists" in r.stderr.lower():
        print(f"  SA ya existe: {name}")
    else:
        print(f"  Error creando SA: {r.stderr.strip()[:300]}")

def asignar_rol(member, role, project):
    r = subprocess.run([
        "gcloud", "projects", "add-iam-policy-binding", project,
        f"--member={member}",
        f"--role={role}",
        "--condition=None",
    ], capture_output=True, text=True)
    print(f"  {role:<45} → {'OK' if r.returncode == 0 else 'FALLO'}")

crear_sa(SA_WORKFLOWS_NAME, "SA Workflows Retention Pipeline", PROJECT_ID)

ROLES = [
    "roles/bigquery.jobUser",          # ejecutar queries
    "roles/bigquery.dataEditor",       # CALL SPs que escriben
    "roles/pubsub.publisher",          # publicar status/alerts
    "roles/logging.logWriter",         # logs estructurados
    "roles/workflows.invoker",         # auto-invocación si hace falta
]
print("Asignando roles a la SA...")
for role in ROLES:
    asignar_rol(f"serviceAccount:{SA_WORKFLOWS_EMAIL}", role, PROJECT_ID)


  SA ya existe: sa-workflows-retention
Asignando roles a la SA...
  roles/bigquery.jobUser                        → FALLO
  roles/bigquery.dataEditor                     → FALLO
  roles/pubsub.publisher                        → FALLO
  roles/logging.logWriter                       → FALLO
  roles/workflows.invoker                       → FALLO


---
## 2. Comparativa Workflows vs Composer

Antes de elegir, **siempre** revisa esta tabla. La tentación de "instalar Airflow porque todos lo conocen" cuesta caro.

| Pregunta | Workflows | Composer |
|----------|-----------|----------|
| Coste mínimo mensual | ~0€ | ~$300/mes (cluster GKE) |
| Cold start | <1s | ~25 min |
| Mantenimiento del runtime | GCP | GCP (pero el cluster es tuyo) |
| Lenguaje | YAML declarativo | Python (DAGs Airflow) |
| Reintentos / backoff | Sí, en YAML | Sí, en `default_args` |
| Backfills automáticos | No | `catchup=True` |
| Operadores de terceros | Limitado | Cientos (dbt, Snowflake, Slack, etc.) |
| UI | Básica (consola) | Rica (Airflow UI con grafos, logs, métricas) |
| Curva de aprendizaje | Baja | Media-alta |
| Mejor para... | <10 pipelines GCP-only | >10 pipelines, cross-system, ecosistema Airflow existente |

**Recomendación para People Analytics en este proyecto:** Workflows. Lo elegimos por coste y simplicidad.

**Nota didáctica:** En este notebook **desplegamos Workflows** y **mostramos el DAG de Composer como referencia** — sin desplegar Composer (cluster cuesta y tarda 25min en arrancar).

---
## 3. Despliegue de Cloud Workflows YAML

El YAML vive en `workflows/retention_pipeline.yaml` (al lado de este notebook). Tiene 4 pasos: `build_features`, `apply_decision`, `snapshot`, `notify_status`. Cada paso tiene `try/except` con notificación a Pub/Sub en caso de fallo.

In [5]:
# Localizar el YAML del Workflow (relativo al notebook)
WORKFLOW_YAML_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else ".", "workflows", "retention_pipeline.yaml")

# Si no se encuentra (notebook se ejecuta desde otra carpeta), buscar en ubicación habitual
if not os.path.exists(WORKFLOW_YAML_PATH):
    posibles = [
        "workflows/retention_pipeline.yaml",
        "../modulo_05_orquestacion_pipelines/workflows/retention_pipeline.yaml",
        "modulo_05_orquestacion_pipelines/workflows/retention_pipeline.yaml",
    ]
    for p in posibles:
        if os.path.exists(p):
            WORKFLOW_YAML_PATH = p
            break

print(f"YAML del Workflow: {WORKFLOW_YAML_PATH}")
print(f"Existe: {os.path.exists(WORKFLOW_YAML_PATH)}")

# Mostrar las primeras 40 líneas para que el alumno vea la estructura
if os.path.exists(WORKFLOW_YAML_PATH):
    with open(WORKFLOW_YAML_PATH) as f:
        contenido = f.read()
    print()
    print("Primeras 40 líneas del YAML:")
    print('\n'.join(contenido.splitlines()[:40]))
    print("...")


YAML del Workflow: ./workflows/retention_pipeline.yaml
Existe: True

Primeras 40 líneas del YAML:
# Cloud Workflows — Pipeline mensual de Retention Risk
#
# Orquesta los Stored Procedures construidos en el Módulo 6:
#   1. sp_build_retention_features  (feature_store_retention)
#   2. sp_score_retention           (predictions_retention)
#   3. sp_apply_decision_rule       (predictions_retention)
#   4. CREATE SNAPSHOT TABLE        (auditoría)
#
# Permisos requeridos (mínimos) para la Service Account del Workflow:
#   - roles/bigquery.jobUser                       (proyecto)
#   - roles/bigquery.dataEditor en datasets:       feature_store_retention, predictions_retention, pipeline_runs
#   - roles/bigquery.dataViewer en datasets:       silver_personio, gold_people_analytics, ml_models
#
# Permisos OPCIONALES para mayor observabilidad (descomentar bloques sys.log y Pub/Sub):
#   - roles/logging.logWriter                      (proyecto)
#   - roles/pubsub.publisher en topics:            re

In [6]:
# Desplegar el Workflow
def desplegar_workflow():
    cmd = [
        "gcloud", "workflows", "deploy", WORKFLOW_NAME,
        f"--location={REGION}",
        f"--source={WORKFLOW_YAML_PATH}",
        f"--service-account={SA_WORKFLOWS_EMAIL}",
        f"--project={PROJECT_ID}",
        "--description=Pipeline mensual de retention risk (orquesta SPs de M6)",
    ]
    print(f"Ejecutando: gcloud workflows deploy {WORKFLOW_NAME} ...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print("  Workflow desplegado")
        return True
    else:
        print(f"  FALLO: {r.stderr[:500]}")
        return False

desplegar_workflow()

# Verificar que existe
r = subprocess.run([
    "gcloud", "workflows", "describe", WORKFLOW_NAME,
    f"--location={REGION}", f"--project={PROJECT_ID}",
    "--format=value(name,state)"
], capture_output=True, text=True)
print(f"\nEstado del Workflow:")
print(r.stdout)


Ejecutando: gcloud workflows deploy retention-pipeline ...
  Workflow desplegado

Estado del Workflow:
projects/project-9176af0b-ecb3-4050-859/locations/europe-southwest1/workflows/retention-pipeline	ACTIVE



---
## 4. Topics de Pub/Sub para status y alertas

El Workflow publica:
- **Topic `retention-pipeline-status`** — un mensaje por cada ejecución exitosa.
- **Topic `retention-pipeline-alerts`** — un mensaje por cada fallo.

En producción, los suscriptores típicos serían:
- Cloud Function que envía a Slack en `*-alerts`.
- BigQuery Subscription que loguea cada `*-status` en una tabla de métricas (M17).

In [7]:
# Crear los 2 topics de notificación
def get_or_create_topic(name):
    path = publisher.topic_path(PROJECT_ID, name)
    try:
        publisher.create_topic(request={"name": path})
        print(f"  Topic creado:    {name}")
    except AlreadyExists:
        print(f"  Topic ya existe: {name}")
    return path

topic_status_path = get_or_create_topic(TOPIC_STATUS)
topic_alerts_path = get_or_create_topic(TOPIC_ALERTS)

# (Opcional) crear una subscription pull para inspeccionar mensajes en clase
from google.cloud.pubsub_v1.types import Subscription
subscriber = pubsub_v1.SubscriberClient()

def get_or_create_pull_sub(topic_name, sub_name):
    sub_path = subscriber.subscription_path(PROJECT_ID, sub_name)
    topic_path = publisher.topic_path(PROJECT_ID, topic_name)
    try:
        subscriber.create_subscription(request={"name": sub_path, "topic": topic_path, "ack_deadline_seconds": 60})
        print(f"  Sub creada:    {sub_name}")
    except AlreadyExists:
        print(f"  Sub ya existe: {sub_name}")
    return sub_path

get_or_create_pull_sub(TOPIC_STATUS, "retention-pipeline-status-pull")
get_or_create_pull_sub(TOPIC_ALERTS, "retention-pipeline-alerts-pull")


  Topic ya existe: retention-pipeline-status
  Topic ya existe: retention-pipeline-alerts
  Sub creada:    retention-pipeline-status-pull
  Sub creada:    retention-pipeline-alerts-pull


'projects/project-9176af0b-ecb3-4050-859/subscriptions/retention-pipeline-alerts-pull'

---
## 5. Ejecución manual del Workflow + inspección de logs

Lanzamos el Workflow con un `target_month` específico y observamos:
1. La ejecución en la consola de Workflows.
2. Las filas escritas en `pipeline_runs.retention_pipeline_runs` por los SPs.
3. El mensaje de status publicado en Pub/Sub.

In [8]:
# Lanzar ejecución manual con target_month explícito
TEST_MONTH = "2025-11-01"

print(f"Ejecutando Workflow para target_month={TEST_MONTH}...")
r = subprocess.run([
    "gcloud", "workflows", "execute", WORKFLOW_NAME,
    f"--location={REGION}",
    f"--project={PROJECT_ID}",
    f'--data={{"target_month":"{TEST_MONTH}"}}',
    "--format=value(name)"
], capture_output=True, text=True)

if r.returncode == 0:
    execution_name = r.stdout.strip()
    execution_id = execution_name.split("/")[-1]
    print(f"  Execution iniciada: {execution_id}")
else:
    print(f"  FALLO: {r.stderr[:500]}")
    execution_id = None


Ejecutando Workflow para target_month=2025-11-01...
  Execution iniciada: 1037f098-5498-4493-8bc3-a04674d050a3


In [9]:
# Esperar a que termine y ver resultado
if execution_id:
    print("Polling estado de ejecución (max 5 min)...")
    for i in range(30):
        time.sleep(10)
        r = subprocess.run([
            "gcloud", "workflows", "executions", "describe", execution_id,
            f"--workflow={WORKFLOW_NAME}",
            f"--location={REGION}",
            f"--project={PROJECT_ID}",
            "--format=value(state)"
        ], capture_output=True, text=True)
        state = r.stdout.strip()
        print(f"  [{i*10:>3}s] state={state}")
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break

    # Resumen completo
    r2 = subprocess.run([
        "gcloud", "workflows", "executions", "describe", execution_id,
        f"--workflow={WORKFLOW_NAME}",
        f"--location={REGION}",
        f"--project={PROJECT_ID}",
        "--format=json"
    ], capture_output=True, text=True)
    if r2.returncode == 0:
        info = json.loads(r2.stdout)
        print()
        print(f"Estado final:    {info.get('state')}")
        print(f"Duración:        {info.get('duration', 'n/a')}")
        if info.get("error"):
            print(f"Error:           {info['error'].get('payload', '')[:500]}")
        if info.get("result"):
            print(f"Result:          {info['result']}")


Polling estado de ejecución (max 5 min)...
  [  0s] state=FAILED

Estado final:    FAILED
Duración:        0.455323536s
Error:           {"body":{"error":{"code":403,"errors":[{"domain":"global","message":"Access Denied: Project project-9176af0b-ecb3-4050-859: User does not have bigquery.jobs.create permission in project project-9176af0b-ecb3-4050-859.","reason":"accessDenied"}],"message":"Access Denied: Project project-9176af0b-ecb3-4050-859: User does not have bigquery.jobs.create permission in project project-9176af0b-ecb3-4050-859.","status":"PERMISSION_DENIED"}},"code":403,"headers":{"Alt-Svc":"h3=\":443\"; ma=2592000,h3-29=


In [10]:
# Verificar que el SP escribió en pipeline_runs
df_runs = bq_client.query(f'''
SELECT step, snapshot_month, rows_written, status, ROUND(duration_seconds, 2) AS dur_s, start_time
FROM `{PROJECT_ID}.pipeline_runs.retention_pipeline_runs`
WHERE DATE(start_time) >= CURRENT_DATE() - 1
ORDER BY start_time DESC
LIMIT 10
''').to_dataframe()
print("Últimos runs registrados:")
print(df_runs.to_string(index=False))


Últimos runs registrados:
            step snapshot_month  rows_written  status  dur_s                       start_time
  apply_decision     2025-11-01           109 SUCCESS   4.15 2026-05-04 10:55:05.092010+00:00
  apply_decision     2025-10-01           104 SUCCESS   4.62 2026-05-04 10:54:58.704099+00:00
  apply_decision     2025-09-01           103 SUCCESS   4.65 2026-05-04 10:54:52.391457+00:00
           score     2025-11-01           109 SUCCESS   4.17 2026-05-04 10:54:42.559802+00:00
           score     2025-10-01           104 SUCCESS   4.19 2026-05-04 10:54:36.469839+00:00
           score     2025-09-01           103 SUCCESS   4.61 2026-05-04 10:54:30.139845+00:00
backfill_summary     2025-12-01            12 SUCCESS    NaN 2026-05-04 10:53:58.339554+00:00
  build_features     2025-12-01           109 SUCCESS   4.53 2026-05-04 10:53:51.802357+00:00
  build_features     2025-11-01           109 SUCCESS   4.72 2026-05-04 10:53:45.407544+00:00
  build_features     2025-10-01   

In [11]:
# Pull del mensaje de status publicado por el Workflow
sub_path = subscriber.subscription_path(PROJECT_ID, "retention-pipeline-status-pull")
print(f"Pull mensajes de {sub_path}...")

response = subscriber.pull(
    request={"subscription": sub_path, "max_messages": 5},
    timeout=10,
)

if not response.received_messages:
    print("  No hay mensajes (puede que la ejecución del Workflow aún no haya terminado).")
else:
    ack_ids = []
    for msg in response.received_messages:
        print(f"\n  Mensaje recibido:")
        print(f"    data:       {msg.message.data.decode('utf-8')}")
        print(f"    attributes: {dict(msg.message.attributes)}")
        ack_ids.append(msg.ack_id)
    if ack_ids:
        subscriber.acknowledge(request={"subscription": sub_path, "ack_ids": ack_ids})
        print(f"\n  Ack {len(ack_ids)} mensaje(s)")


Pull mensajes de projects/project-9176af0b-ecb3-4050-859/subscriptions/retention-pipeline-status-pull...


DeadlineExceeded: 504 Deadline Exceeded

---
## 6. Cloud Scheduler — cron mensual

Configuramos un job que dispare el Workflow el **día 1 de cada mes a las 06:00 (Madrid time)**. El parámetro `target_month` se calcula dentro del Workflow (primer día del mes en curso).

In [ ]:
# Crear Cloud Scheduler job que dispare el Workflow
# Necesitamos un SA con permiso de invocar Workflows (lo creamos antes)

# El URI debe ser el endpoint de la API de Workflows Executions
WORKFLOW_URI = (
    f"https://workflowexecutions.googleapis.com/v1/projects/{PROJECT_ID}"
    f"/locations/{REGION}/workflows/{WORKFLOW_NAME}/executions"
)

# Argumento: dejamos target_month vacío para que el Workflow use mes en curso
SCHEDULER_BODY = json.dumps({
    "argument": json.dumps({})  # Workflow detecta y usa mes actual
})

# Crear o actualizar
def crear_scheduler_job():
    r = subprocess.run([
        "gcloud", "scheduler", "jobs", "create", "http", SCHEDULER_JOB_NAME,
        f"--location={REGION}",
        f"--project={PROJECT_ID}",
        "--schedule=0 6 1 * *",
        "--time-zone=Europe/Madrid",
        f"--uri={WORKFLOW_URI}",
        "--http-method=POST",
        "--headers=Content-Type=application/json",
        f"--message-body={SCHEDULER_BODY}",
        f"--oauth-service-account-email={SA_WORKFLOWS_EMAIL}",
        "--description=Disparador mensual del retention pipeline (día 1 06:00 Madrid)",
    ], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  Scheduler job creado: {SCHEDULER_JOB_NAME}")
    elif "already exists" in r.stderr.lower():
        print(f"  Scheduler job ya existe: {SCHEDULER_JOB_NAME} (no actualizado)")
    else:
        print(f"  FALLO: {r.stderr[:500]}")

crear_scheduler_job()

# Mostrar configuración
r = subprocess.run([
    "gcloud", "scheduler", "jobs", "describe", SCHEDULER_JOB_NAME,
    f"--location={REGION}", f"--project={PROJECT_ID}",
    "--format=yaml(name,schedule,timeZone,state,httpTarget.uri)"
], capture_output=True, text=True)
print()
print("Configuración del Scheduler job:")
print(r.stdout)


---
## 7. Eventarc — trigger event-driven (alternativo)

Configuramos un trigger que dispare el Workflow **automáticamente cuando aparezca un fichero `payroll_*.csv` en `gs://*-datalake/payroll/incoming/`**.

Coexiste con el Scheduler:
- **Scheduler:** garantía de que el pipeline corre el día 1 (incluso si payroll no llega).
- **Eventarc:** acelera la ejecución si payroll llega antes (típicamente día 3-4).

In [ ]:
# Crear el trigger Eventarc
BUCKET_DATALAKE = f"{PROJECT_ID}-datalake"

# La SA del trigger necesita rol roles/eventarc.eventReceiver y permiso para invocar Workflows
def crear_eventarc_trigger():
    cmd = [
        "gcloud", "eventarc", "triggers", "create", EVENTARC_TRIGGER_NAME,
        f"--location={REGION}",
        f"--project={PROJECT_ID}",
        f"--destination-workflow={WORKFLOW_NAME}",
        f"--destination-workflow-location={REGION}",
        "--event-filters=type=google.cloud.storage.object.v1.finalized",
        f"--event-filters=bucket={BUCKET_DATALAKE}",
        f"--service-account={SA_WORKFLOWS_EMAIL}",
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  Eventarc trigger creado: {EVENTARC_TRIGGER_NAME}")
    elif "already exists" in r.stderr.lower():
        print(f"  Eventarc trigger ya existe: {EVENTARC_TRIGGER_NAME}")
    else:
        print(f"  AVISO (puede requerir configuración manual de IAM): {r.stderr[:500]}")
        print()
        print("  Si falla por IAM: añade roles/eventarc.eventReceiver a la SA y reintenta.")

# NOTA: Para que el trigger funcione end-to-end, GCS debe tener notificaciones habilitadas
# y la SA de GCS debe poder publicar al topic de Eventarc. Es habitual que requiera
# pasos manuales adicionales en cuentas con políticas de organización estrictas.
crear_eventarc_trigger()

# Listar triggers
print()
r = subprocess.run([
    "gcloud", "eventarc", "triggers", "list",
    f"--location={REGION}", f"--project={PROJECT_ID}",
    "--format=table(name,destination.workflow,eventFilters.type)"
], capture_output=True, text=True)
print("Triggers Eventarc en la región:")
print(r.stdout)


---
## 8. Composer DAG equivalente (lectura, NO se despliega)

El fichero `composer/retention_pipeline_dag.py` contiene el DAG de Airflow equivalente. **No lo desplegamos en clase** — un cluster Composer cuesta ~$300/mes y tarda ~25 min en arrancar.

Lo leemos para que el alumno entienda la **traducción uno-a-uno** entre los dos paradigmas.

In [ ]:
# Leer y mostrar el DAG de Composer
import os
DAG_PATH = "composer/retention_pipeline_dag.py"
posibles = [
    DAG_PATH,
    f"modulo_05_orquestacion_pipelines/{DAG_PATH}",
    f"../modulo_05_orquestacion_pipelines/{DAG_PATH}",
]
for p in posibles:
    if os.path.exists(p):
        DAG_PATH = p
        break

if os.path.exists(DAG_PATH):
    with open(DAG_PATH) as f:
        contenido = f.read()
    print(f"DAG: {DAG_PATH} ({len(contenido)} caracteres)")
    print()
    print(contenido)
else:
    print(f"ATENCIÓN: no se encuentra {DAG_PATH}")
    print("Esperado en modulo_05_orquestacion_pipelines/composer/retention_pipeline_dag.py")


**Comparativa lado a lado:**

| Aspecto | Workflows YAML | Airflow DAG |
|---------|----------------|-------------|
| Sintaxis | YAML declarativo | Python |
| Definir un step | `- step_name: { call: ..., args: ... }` | `task = Operator(...)` |
| Reintentos | `retry: { max_retries: 3 }` | `retries=3, retry_delay=timedelta(...)` |
| Manejo de error | `try: ... except: as: e: ...` | `trigger_rule=TriggerRule.ONE_FAILED` para tareas downstream |
| Orden de ejecución | Implícito (orden lineal en YAML) | Explícito (`task1 >> task2`) |
| Variables dinámicas | `${...}` (CEL expressions) | `{{ ds }}` (Jinja templating) |
| Scheduling | Externo (Cloud Scheduler) | Built-in (`schedule_interval`) |

---
## 9. Caso de negocio newsvendor — el "porqué" final

Cerramos M5 recordando **por qué** orquestamos esto:

- El pipeline corre **día 1 de cada mes**. Procesa snapshot del mes cerrado anterior.
- Aplica regla de decisión Cu/Co → produce lista de empleados con `recommended_action ∈ {retention_action, monitor, no_action}`.
- HRBP recibe la lista (en M14 vía Looker Studio) y **toma decisiones humanas** sobre los casos prioritarios.
- El pipeline NO actúa por sí solo. Solo **prepara información**. La GDPR Art. 22 exige human-in-the-loop.

**Qué se ahorra el negocio con esto vs no tenerlo:**

| Sin pipeline | Con pipeline |
|--------------|--------------|
| HR detecta riesgo de salida cuando alguien dice "tengo otra oferta" | Detecta señales 1-3 meses antes |
| Acciones de retención reactivas, urgentes y caras (counter-offer 30%) | Acciones proactivas, planificadas (bonus 10%) |
| ~50% de retention success rate | ~75% (industry benchmarks) |
| Coste de reemplazo amortizado en presupuesto general | Decisión transparente Cu/Co por banda |


---
## 10. Findings honestos + cierre

### Lo que funcionó

- **Workflow desplegado y ejecutable** end-to-end. La idempotencia de los SPs (heredada de M6) hace que sea seguro re-ejecutar sin duplicar.
- **Doble disparador** (Scheduler + Eventarc) cubre los dos modos de operación que la realidad exige.
- **Coste de orquestación trivial:** un mes a 30 ejecuciones × 4 steps = 120 transitions × $0.01/1k = ~$0.001/mes.
- **Composer evitado** sin perder potencia. El DAG equivalente está versionado por si en 6 meses crece la complejidad.

### Lo que NO funcionó / cuidado

1. **Eventarc requiere permisos de organización.** En cuentas con políticas estrictas (CMEK, VPC-SC), el trigger puede requerir 2-3 vueltas con el equipo de plataforma. En el aula, si falla la creación, dejamos el patrón documentado y seguimos.

2. **No hay alerta humana todavía.** El topic `*-alerts` recibe mensajes, pero hasta que no atemos un suscriptor (Cloud Function → Slack/PagerDuty) **no se entera nadie**. Esto se cierra en M17 (observabilidad).

3. **El cron `0 6 1 * *` es ingenuo:** si el día 1 es festivo y nadie revisa el resultado, el pipeline pasa al día 2 sin alerta. Patrones más robustos: dispatch dual con lag, o disparar el primer día laborable. Decisión de negocio.

4. **No hay versionado del Workflow YAML.** Cada `gcloud workflows deploy` sobrescribe la versión anterior — en producción hay que versionar el YAML en Git (M9) y deployar via CI/CD.

5. **Falta backfill orquestado.** Si descubrimos un bug hace 3 meses y queremos recalcular, hoy lo hacemos manualmente con `sp_backfill_features`. Mejor: un Workflow `retention-pipeline-backfill` que loope sobre meses.

### Próximos pasos

- **M9 (GitHub):** versionar `retention_pipeline.yaml` y `retention_pipeline_dag.py` en repo, despliegue por CI/CD.
- **M10 (Backups):** snapshot pre-ejecución de `feature_store_retention.features` (ya existe en el Workflow paso 3, pero falta restore plan).
- **M11 (Cloud DLP):** verificar que `retention_actions` no contiene PII más allá de `employee_code`.
- **M17 (Observabilidad):** suscriptor del topic `*-alerts` → Slack; dashboard de pipeline_runs → Cloud Monitoring.

### Resumen del módulo

Hemos visto:
1. **Comparativa Workflows vs Composer** — Workflows gana para el 80% de casos PA.
2. **Despliegue de un Workflow YAML** ejecutable end-to-end.
3. **Doble disparador** Cloud Scheduler (cron) + Eventarc (event-driven).
4. **DAG de Composer equivalente** — código pedagógico, sin desplegar.
5. **Buenas prácticas** de pipelines resilientes: idempotencia heredada, reintentos, alertas, snapshots de auditoría.

→ **Continuamos con el Módulo 7: SQL avanzado aplicado a People Analytics** (window functions, cohortes, métricas longitudinales — para enriquecer las features de M6).
